# Problem regresyjny

## Przewidywanie cen mieszkań/domów w Kalifornii

### Import bibliotek (sklearn tylko do pobrania danych)

In [3]:
import numpy as np
import random
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

### Wczytanie danych

In [21]:
california = fetch_california_housing()
X = california.data
y = california.target

y

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,))

### Skalowanie danych - pewnie później zrobi sie to bez biblioteki

In [8]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Podział na zbiór trenignowy i testowy

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

### Formatowanie danych na listy krotek, aby były kompatybilne z siecią

In [13]:
training_data = [(x.reshape(-1, 1), np.array([[y_val]])) for x, y_val in zip(X_train, y_train)]
test_data = [(x.reshape(-1, 1), np.array([[y_val]])) for x, y_val in zip(X_test, y_test)]

print(f"Dane gotowe! Treningowe: {len(training_data)}, Testowe: {len(test_data)}")
print(f"Kształt wejścia: {training_data[0][0].shape}, Kształt wyjścia: {training_data[0][1].shape}")

Dane gotowe! Treningowe: 16512, Testowe: 4128
Kształt wejścia: (8, 1), Kształt wyjścia: (1, 1)


## Narzędzia Matematyczne

In [14]:
def sigmoid(z):
    """Funkcja aktywacji dla warstw ukrytych."""
    # np.clip zapobiega przepełnieniu (overflow) przy dużych liczbach
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_prime(z):
    """Pochodna funkcji sigmoid."""
    return sigmoid(z) * (1 - sigmoid(z))

class MeanSquaredError:
    @staticmethod
    def fn(a, y):
        """Zwraca błąd średniokwadratowy MSE."""
        return 0.5 * np.linalg.norm(a - y)**2

    @staticmethod
    def delta(a, y):
        """
        Zwraca błąd (deltę) dla warstwy WYJŚCIOWEJ LINIOWEJ.
        Dla funkcji liniowej f(z) = z, pochodna to 1.
        Zatem delta = pochodna_kosztu * 1 = (a - y)
        """
        return (a - y)

## Właściwa Sieć Perceptronowa

In [15]:
class RegressionNetwork:
    def __init__(self, sizes):
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.cost = MeanSquaredError
        
        # Ulepszona inicjalizacja wag (jak w network2.py)
        self.biases = [np.random.randn(y, 1) for y in self.sizes[1:]]
        self.weights = [np.random.randn(y, x) / np.sqrt(x) 
                        for x, y in zip(self.sizes[:-1], self.sizes[1:])]

    def feedforward(self, a):
        # 1. Przejście przez warstwy ukryte (z Sigmoidą)
        for b, w in zip(self.biases[:-1], self.weights[:-1]):
            a = sigmoid(np.dot(w, a) + b)
            
        # 2. Przejście przez warstwę wyjściową (LINIOWA - brak sigmoidy!)
        a = np.dot(self.weights[-1], a) + self.biases[-1]
        return a

    def SGD(self, training_data, epochs, mini_batch_size, eta, test_data=None):
        n = len(training_data)
        for j in range(epochs):
            random.shuffle(training_data)
            mini_batches = [training_data[k:k+mini_batch_size] for k in range(0, n, mini_batch_size)]
            
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta)
            
            if test_data:
                mse, mae = self.evaluate(test_data)
                print(f"Epoka {j+1}: Błąd średni (MAE) wynosi ~${mae*100000:,.0f} (MSE: {mse:.4f})")
            else:
                print(f"Epoka {j+1} zakończona")

    def update_mini_batch(self, mini_batch, eta):
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        
        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)
            nabla_b = [nb + dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw + dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]
            
        self.weights = [w - (eta / len(mini_batch)) * nw for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b - (eta / len(mini_batch)) * nb for b, nb in zip(self.biases, nabla_b)]

    def backprop(self, x, y):
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        
        activation = x
        activations = [x]
        zs = []
        
        # Przejście w przód - WARSTWY UKRYTE
        for b, w in zip(self.biases[:-1], self.weights[:-1]):
            z = np.dot(w, activation) + b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)
            
        # Przejście w przód - OSTATNIA WARSTWA (liniowa)
        z_out = np.dot(self.weights[-1], activation) + self.biases[-1]
        zs.append(z_out)
        activation = z_out # Brak sigmoidy
        activations.append(activation)
        
        # Propagacja wsteczna (Ostatnia warstwa)
        delta = self.cost.delta(activations[-1], y)
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].transpose())
        
        # Propagacja wsteczna (Warstwy ukryte)
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            delta = np.dot(self.weights[-l+1].transpose(), delta) * sp
            nabla_b[-l] = delta
            nabla_w[-l] = np.dot(delta, activations[-l-1].transpose())
            
        return (nabla_b, nabla_w)

    def evaluate(self, test_data):
        # MAE: Średni błąd bezwzględny (bardziej intuicyjny dla cen)
        # MSE: Błąd średniokwadratowy
        total_mae = 0.0
        total_mse = 0.0
        
        for x, y in test_data:
            prediction = self.feedforward(x)[0][0]
            actual = y[0][0]
            
            total_mae += abs(prediction - actual)
            total_mse += (prediction - actual)**2
            
        n = len(test_data)
        return total_mse / n, total_mae / n

## Pierwsza Nauka

In [29]:
# Mamy 8 cech na wejściu, damy 20 neuronów ukrytych, i 1 neuron ceny na wyjściu
net = RegressionNetwork([8, 20, 1])

# Uczymy małym krokiem (eta = 0.01), bo regresja liniowa na końcu potrafi mocno "skakać"
print("Rozpoczynam wycenę nieruchomości...")
net.SGD(training_data, epochs=30, mini_batch_size=10, eta=0.01, test_data=test_data)

Rozpoczynam wycenę nieruchomości...
Epoka 1: Błąd średni (MAE) wynosi ~$57,120 (MSE: 0.5774)
Epoka 2: Błąd średni (MAE) wynosi ~$51,950 (MSE: 0.5271)
Epoka 3: Błąd średni (MAE) wynosi ~$53,933 (MSE: 0.5153)
Epoka 4: Błąd średni (MAE) wynosi ~$50,360 (MSE: 0.4935)
Epoka 5: Błąd średni (MAE) wynosi ~$50,939 (MSE: 0.4889)
Epoka 6: Błąd średni (MAE) wynosi ~$51,053 (MSE: 0.4840)
Epoka 7: Błąd średni (MAE) wynosi ~$49,331 (MSE: 0.4776)
Epoka 8: Błąd średni (MAE) wynosi ~$48,639 (MSE: 0.4760)
Epoka 9: Błąd średni (MAE) wynosi ~$48,552 (MSE: 0.4716)
Epoka 10: Błąd średni (MAE) wynosi ~$48,106 (MSE: 0.4700)
Epoka 11: Błąd średni (MAE) wynosi ~$50,323 (MSE: 0.4705)
Epoka 12: Błąd średni (MAE) wynosi ~$47,601 (MSE: 0.4644)
Epoka 13: Błąd średni (MAE) wynosi ~$47,914 (MSE: 0.4571)
Epoka 14: Błąd średni (MAE) wynosi ~$47,508 (MSE: 0.4555)
Epoka 15: Błąd średni (MAE) wynosi ~$47,247 (MSE: 0.4530)
Epoka 16: Błąd średni (MAE) wynosi ~$48,131 (MSE: 0.4531)
Epoka 17: Błąd średni (MAE) wynosi ~$47,613 (

## Test dla przykładowego wejścia

In [28]:
# Wybieramy konkretny dom (np. dom nr 10 ze zbioru testowego)
index = 67
raw_x = X_test[index] 
actual_price = y_test[index]

# Odwracamy normalizację, żeby zobaczyć PRAWDZIWE liczby, a nie te od -1 do 1
real_features = scaler.inverse_transform([raw_x])[0]
feature_names = california.feature_names

print(f"--- PARAMETRY NIERUCHOMOŚCI NR {index} ---")
# zip łączy nazwy cech z ich wartościami, żebyśmy mogli wypisać je w pętli
for name, val in zip(feature_names, real_features):
    # Formatujemy wyświetlanie: 10 znaków na nazwę, 4 miejsca po przecinku dla wartości
    print(f"{name:10}: {val:.4f}")

print("-" * 35)

# Przekazujemy znormalizowany wektor do sieci (musi mieć kształt kolumnowy 8x1)
x_for_network = raw_x.reshape(-1, 1)
predicted_price = net.feedforward(x_for_network)[0][0]

# Mnożymy przez 100 000, bo wartości y są w setkach tysięcy dolarów
print(f"Prawdziwa cena rynkowa: ${actual_price * 100000:,.0f}")
print(f"Wycena Twojej sieci   : ${predicted_price * 100000:,.0f}")
print(f"Różnica w wycenie     : ${abs(actual_price - predicted_price) * 100000:,.0f}")

--- PARAMETRY NIERUCHOMOŚCI NR 67 ---
MedInc    : 4.0767
HouseAge  : 5.0000
AveRooms  : 5.4605
AveBedrms : 1.0889
Population: 3207.0000
AveOccup  : 3.1690
Latitude  : 38.0300
Longitude : -121.9500
-----------------------------------
Prawdziwa cena rynkowa: $143,100
Wycena Twojej sieci   : $173,962
Różnica w wycenie     : $30,862


### Szukanie parametrów / dobieranie zmiennych / może jakiś cross validation idk